# NLP — Text Analysis (POS, NER, Reviews)

## الترتيب / Flow
1. استيراد المكتبات - Import libraries
2. POS tagging - Part-of-speech
3. Dependency parsing highlights
4. Named Entity Recognition (NER)
5. Full SpaCy pipeline function
6. مشروع تحليل المراجعات - Product review analyzer

In [ ]:
# Step 1) استيراد المكتبات / Import libraries
# pip install spacy pandas -q
# python -m spacy download en_core_web_sm
import pandas as pd
import spacy
from collections import Counter, defaultdict

nlp = spacy.load('en_core_web_sm')

In [ ]:
# Step 2) POS tagging / تصنيف الكلمات نحوياً
text = "Apple CEO Tim Cook announced record quarterly revenue in California."
doc = nlp(text)

print(f"{'Token':<12} {'POS':<8} {'Tag':<8} {'Dep'}")
for token in doc:
    print(f"{token.text:<12} {token.pos_:<8} {token.tag_:<8} {token.dep_}")

In [ ]:
# Step 3) Dependency parsing / العلاقات النحوية
print("Noun phrases:")
for chunk in doc.noun_chunks:
    print(f"  - {chunk.text}")

print("\nMain verbs:")
for token in doc:
    if token.pos_ == 'VERB':
        subj = [c.text for c in token.children if c.dep_ in ('nsubj', 'nsubjpass')]
        print(f"  {subj or ['?']} -> {token.lemma_}")

In [ ]:
# Step 4) NER / استخراج الكيانات
news = """
NVIDIA CEO Jensen Huang unveiled the H200 GPU at GTC in San Jose, California.
Microsoft and Google announced orders worth billions. Revenue hit $22 billion.
"""
doc = nlp(news)

entities = defaultdict(set)
for ent in doc.ents:
    entities[ent.label_].add(ent.text)

for label, ents in sorted(entities.items()):
    print(f"{label}: {', '.join(sorted(ents))}")

In [ ]:
# Step 5) Full pipeline / خط أنابيب كامل
def analyze_text(text: str) -> dict:
    doc = nlp(text)
    clean_tokens = [t.text for t in doc if not t.is_stop and not t.is_punct and t.is_alpha]
    lemmas = [t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct and t.is_alpha]
    return {
        'word_count': len([t for t in doc if not t.is_punct and not t.is_space]),
        'clean_tokens': clean_tokens,
        'lemmas': lemmas,
        'pos_counts': dict(Counter(t.pos_ for t in doc if not t.is_punct)),
        'entities': [(e.text, e.label_) for e in doc.ents],
        'noun_phrases': [c.text for c in doc.noun_chunks],
    }

result = analyze_text("Tesla reported revenue growth in Austin, Texas.")
for key, value in result.items():
    print(f"{key}: {value}")

In [ ]:
# Step 6) مشروع تحليل المراجعات / Product review analyzer
reviews_df = pd.read_csv('product_reviews.csv')
reviews_df.head()

In [ ]:
POSITIVE = {'amazing', 'incredible', 'fantastic', 'excellent', 'outstanding', 'fast', 'happy', 'worth', 'best'}
NEGATIVE = {'terrible', 'useless', 'cracked', 'cheap', 'mediocre', 'disappointing', 'barely'}
ASPECTS = {
    'camera': ['camera', 'photo', 'photography'],
    'battery': ['battery', 'charge', 'power', 'lasts'],
    'screen': ['screen', 'display'],
    'price': ['price', 'cost', 'expensive', 'cheap', 'worth'],
    'performance': ['fast', 'speed', 'slow', 'processor'],
}

def analyze_review(row):
    doc = nlp(row['review'])
    lemmas = [t.lemma_.lower() for t in doc]
    adj = [t.lemma_.lower() for t in doc if t.pos_ == 'ADJ']
    aspects = [a for a, kws in ASPECTS.items() if any(k in lemmas for k in kws)]
    return {
        'product': row['product'],
        'rating': row['rating'],
        'positive': [w for w in adj if w in POSITIVE],
        'negative': [w for w in adj if w in NEGATIVE],
        'aspects': aspects,
        'entities': [(e.text, e.label_) for e in doc.ents],
    }

for _, row in reviews_df.iterrows():
    r = analyze_review(row)
    print(f"\n{r['product']} ({r['rating']}/5)")
    print(f"  + {r['positive'] or ['none']}")
    print(f"  - {r['negative'] or ['none']}")
    print(f"  aspects: {r['aspects'] or ['none']}")